# Your First SLURM Pipeline

## What you'll learn

- Run built-in Artisan operations using `SlurmRunner` as the pipeline default
- Connect generated CSVs to a transformation and metric calculation
- Check the actual output data with public inspection APIs

**Prerequisites:** Complete the [tutorial setup](index.md). Your kernel needs
`sbatch`, `squeue`, and `sacct` access, a shared Python environment, and a
shared output directory. Set your partition and optional account before
starting Jupyter. This notebook submits real jobs.
**Estimated time:** 10 minutes, plus queue time.
**GPU required:** No.

## Import the installed packages

The workflow uses operations shipped with `artisan`; it needs no source
fixtures or custom worker modules. Check the printed versions and import
paths before submitting work.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import artisan
import polars as pl
from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import (
    inspect_data,
    inspect_metrics,
    inspect_pipeline,
    inspect_step,
)

import artisan_submitit
from artisan_submitit import SlurmRunner

print(f"Artisan {artisan.__version__}: {Path(artisan.__file__).resolve()}")
print(
    f"artisan-submitit {artisan_submitit.__version__}: {Path(artisan_submitit.__file__).resolve()}"
)

## Choose a shared workspace

`ARTISAN_TUTORIAL_ROOT` must name a directory visible at the same absolute path
on the manager and compute nodes. The default, `tutorial-runs` under the
notebook's working directory, is suitable only when that location is shared.
Data, staging, working files, and logs stay beneath this tutorial's directory.

The next cell calls `tutorial_setup(..., clean=True)`, which deletes this
notebook's previous `runs/first-slurm-pipeline` directory before creating a fresh store.
Finish or cancel previous jobs before rerunning setup. Save results you want
to keep elsewhere, then restart the kernel and run all cells in order.

In [ ]:
shared_root = Path(
    os.environ.get("ARTISAN_TUTORIAL_ROOT", str(Path.cwd() / "tutorial-runs"))
).expanduser()
if not shared_root.is_absolute():
    message = "ARTISAN_TUTORIAL_ROOT must be an absolute shared path"
    raise ValueError(message)
shared_root = shared_root.resolve()

env = tutorial_setup("first-slurm-pipeline", base_dir=shared_root, clean=True)


partition = os.environ.get("ARTISAN_SLURM_PARTITION", "cpu")
account = os.environ.get("ARTISAN_SLURM_ACCOUNT")
options = {"slurm_account": account} if account else {}
runner = SlurmRunner(
    slurm_partition=partition,
    log_folder=str(Path(env.runs_dir) / "logs"),
    **options,
)
print(f"Partition: {partition}; account: {account or 'site default'}")


resources = {"cpus": 1, "memory_gb": 1, "time_limit": "00:05:00"}
print(f"Tutorial results: {env.runs_dir}")

## Generate, transform, and measure

The pipeline default sends all three steps to SLURM. Each task requests one
CPU, 1 GiB of memory, and five minutes. `max_workers=2` limits concurrent
worker tasks within each step. Steps here run sequentially because `run()`
waits for its result.

`DataGenerator` creates four CSV files with ten rows each. `DataTransformer`
halves the numeric values without noise and preserves the row IDs.
`MetricCalculator` produces one metric artifact for each transformed file.
Note the generator's plural output role, `datasets`; the transform uses
`dataset` for both input and output.

The `finally` block closes the pipeline even if a step raises an exception.

In [ ]:
pipeline = PipelineManager.create(
    name="first_slurm_pipeline",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
)
output = pipeline.output

try:
    generated = pipeline.run(
        DataGenerator,
        name="generate",
        params={"count": 4, "rows_per_file": 10, "seed": 42},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
    transformed = pipeline.run(
        DataTransformer,
        name="transform",
        inputs={"dataset": output("generate", "datasets")},
        params={"scale_factor": 0.5, "noise_amplitude": 0, "variants": 1, "seed": 100},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
    measured = pipeline.run(
        MetricCalculator,
        name="metrics",
        inputs={"dataset": output("transform", "dataset")},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
finally:
    summary = pipeline.finalize()

assert summary["overall_success"], summary
assert summary["total_steps"] == 3
run_id = pipeline.config.pipeline_run_id
inspect_pipeline(env.delta_root, pipeline_run_id=run_id)

## Check artifacts and transformed values

Select this run explicitly when reading accepted outputs. Each step should
expose four artifacts. The generator's `succeeded_count` is one because
a single generative execution produced all four files; it is not an
artifact count or a scheduler job count.

The following assertions compare every transformed numeric value with its
source, including the operation's four-decimal rounding. The `_source` column
contains artifact names without the `.csv` extension.

In [ ]:
for step in (generated, transformed, measured):
    artifacts = inspect_step(env.delta_root, step.step_number, pipeline_run_id=run_id)
    assert artifacts.height == 4, (step.step_name, artifacts)
assert generated.succeeded_count == 1

source_data = inspect_data(
    env.delta_root, step_number=generated.step_number, pipeline_run_id=run_id
)
transformed_data = inspect_data(
    env.delta_root, step_number=transformed.step_number, pipeline_run_id=run_id
)
assert source_data.height == transformed_data.height == 40
assert source_data["_source"].n_unique() == transformed_data["_source"].n_unique() == 4

for name in sorted(source_data["_source"].unique()):
    source = source_data.filter(pl.col("_source") == name).sort("id")
    actual = transformed_data.filter(pl.col("_source") == f"{name}_0").sort("id")
    assert source.height == actual.height == 10
    assert source["id"].to_list() == actual["id"].to_list()
    for column in ("x", "y", "z", "score"):
        expected = [round(value * 0.5, 4) for value in source[column]]
        assert all(
            abs(found - wanted) < 1e-8
            for found, wanted in zip(actual[column], expected, strict=True)
        ), (name, column)

transformed_data.head()

## Read the metrics

There should be four metric artifacts, each describing a ten-row dataset.
The table also shows distribution statistics for the transformed scores.

In [ ]:
metrics = inspect_metrics(
    env.delta_root, step_number=measured.step_number, pipeline_run_id=run_id
)
assert metrics.height == 4
assert metrics["summary.row_count"].to_list() == [10, 10, 10, 10]
metrics

## Next steps

- [Inside an allocation](02-inside-an-allocation.ipynb) runs the same workflow
  using `srun` within an allocation you already own.
- [Batching and managed commands](03-batching-and-commands.ipynb) groups
  inputs into execution units and worker tasks.
- [Queue execution](../slurm.md) explains resource configuration, array
  waves, logs, and cancellation evidence.

To keep a later preparation step on the manager, import
`Runner` from `artisan.orchestration` and pass `step_runner=Runner.LOCAL`
to that `pipeline.run()` call. The remaining steps retain the pipeline's
`SlurmRunner` default.